## 1. LLM 최적화 전략별 종합 트레이드오프 (Trade-off)

실무에서 프로젝트의 요구 사항에 따라 어떤 기법을 선택해야 하는지 결정하기 위한 핵심 기준은 다음과 같습니다.

| 비교 항목 | 프롬프트 엔지니어링 (Prompt Eng.) | RAG (검색 증강 생성) | PEFT (LoRA 등) | 풀 파인튜닝 (Full FT) | RLHF / DPO |
|:---|:---|:---|:---|:---|:---|
| **필요 데이터량** | 매우 소량 (수 개) | 사내 지식 문서 데이터 | 소~중형 (수백~수만 개) | 대량 (수만~수백만 개) | 선호도 데이터 (수천 개) |
| **GPU 학습 리소스** | 필요 없음 (0) | 필요 없음 (0) | 소형 GPU 1대로 가능 | 수십~수백 대 GPU 필요 | 중~대형 GPU 클러스터 필요 |
| **초기 개발 난이도**| 매우 낮음 | 보통 (인덱싱 필요) | 보통 (코드 구현 필요) | 높음 (인프라 필요) | 매우 높음 (정렬 피드백) |
| **지식 갱신 주기** | 즉시 (프롬프트 변경) | 실시간 (DB 업데이트) | 재학습 주기 필요 | 매우 긴 재학습 주기 | 매우 긴 재학습 주기 |
| **주요 목적** | 빠른 검증, 일반 질의 | 사내 문서 기반 사실 답변 | 스타일, 어조, 도메인 적응 | 기초 모델 개발, 도메인 특화 | 인간 윤리 및 선호 정렬 |

In [1]:
import time
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cpu'

## 2. 평가 시나리오: 회사 FAQ 답변 평가
사내 환불 규정 및 요금 정책에 대한 사용자 질문에 대응하는 세 가지 아키텍처 모델의 출력을 준비하고, 이를 **Ground Truth(정답 가이드)** 와 비교하여 정량 평가해 봅니다.

- **시나리오 구성**:
  1. **Ground Truth**: 사내 가이드 상의 모범 답안
  2. **Base Model (gpt2)**: 사내 정보가 학습되지 않아 일반 웹 문서 스타일로 답변하는 기본 모델 출력
  3. **RAG Model**: 관련 내부 문서를 프롬프트 컨텍스트에 동봉하여 사실 기반으로 답변하는 아키텍처 출력
  4. **Fine-tuned Model**: 사내 FAQ 데이터를 학습하여 간결하게 양식에 맞춰 답변하는 어댑터 모델 출력

In [2]:
# 평가용 데이터 정의
eval_faq_data = [
    {
        "question": "What is the company refund policy?",
        "ground_truth": "Customers can request a full refund within 14 days of purchase if the service is unused.",
        "base_model_output": "Refund policy varies depending on the country. Please check our global website for more details or contact sales.",
        "rag_model_output": "According to our official guide, you can get a full refund within 14 days of purchase as long as the service remains unused.",
        "finetuned_model_output": "Refund policy: Full refund within 14 days of purchase for unused services."
    },
    {
        "question": "Is there an extra charge for weekend support?",
        "ground_truth": "No, standard weekend email support is included in all premium plans without extra charges.",
        "base_model_output": "Support services are usually provided on weekdays. Weekend calls might be billed per hour.",
        "rag_model_output": "Standard weekend email support is included in premium plans with no extra charges.",
        "finetuned_model_output": "Weekend support charges: standard email support is free of charge on premium plans."
    }
 ]
df_eval = pd.DataFrame(eval_faq_data)
print("Evaluation dataset initialized!")

Evaluation dataset initialized!
